In [1]:
import pandas as pd
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [ ]:
df = pd.read_parquet('/workspaces/mlops/data/yellow_tripdata_2023-01.parquet')
df.shape[1]

In [ ]:
df.head()

In [ ]:
df['duration'] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds() / 60
print(df['duration'].std())

In [ ]:
df_filtered = df[(df['duration'] >= 1) & (df['duration'] <= 60)]

In [ ]:
fraction = len(df_filtered) / len(df)
print(f"{fraction:.2%}")

In [ ]:
df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]
categorical = ['PULocationID', 'DOLocationID']
df[categorical] = df[categorical].astype(str)

train_dicts = df[categorical].to_dict(orient = 'records')

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

print(X_train.shape[1])

In [ ]:
y_train = df['duration'].values

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_train)
rmse = root_mean_squared_error(y_train, y_pred)
print(rmse)

In [ ]:
df_val = pd.read_parquet('/workspaces/mlops/data/yellow_tripdata_2023-02.parquet')
df_val['duration'] = (
    df_val.tpep_dropoff_datetime - df_val.tpep_pickup_datetime
).dt.total_seconds() / 60
df_val = df_val[(df_val.duration >= 1) & (df_val.duration <= 60)]

df_val[categorical] = df_val[categorical].astype(str)
val_dicts = df_val[categorical].to_dict(orient='records')

X_val = dv.transform(val_dicts)
y_val = df_val['duration'].values

y_pred = lr.predict(X_val)
rmse_val = root_mean_squared_error(y_val, y_pred)

print(rmse_val)